In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
import joblib
import gc
import numpy as np
from slum_segmentation import helper_functions, ml_dataset, np_dataset
from sklearn.ensemble import RandomForestClassifier

# Load and shuffle training and testing data. Use `20%` data from the test set for progressive evaluation durting training

In [3]:
def shuffle_set(x, y, rng):
    permutation = rng.permutation(len(y))
    return x[permutation], y[permutation]
    
rng = np.random.default_rng(42)

In [4]:
x_train, y_train = ml_dataset.cities.combined.get_train_data()
x_train, y_train = shuffle_set(x_train, y_train, rng)

In [4]:
x_val, y_val = ml_dataset.cities.combined.get_val_data()
x_val, y_val = shuffle_set(x_val, y_val, rng)

# Train random forest

In [ ]:
# prevent the "class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start 
# if the fitted data differs from the full dataset" user warning as it does not apply in out case
warnings.filterwarnings(
    "ignore",
    message='class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start.*',
    category=UserWarning
)

rf = RandomForestClassifier(
    n_estimators=50, # start with 50, increase till 200
    bootstrap=True, # give a random subset of data to each tree with replacement
    class_weight="balanced", # class weight will be inversely proportional to frequency, thus informal settlement class pixels will be given more weight 
    max_features=5, # only 5 random features will be considered at each split
    max_depth=20, # stopping condition on tree size
    min_samples_leaf=10, # minimum of 10 samples at each leaf node
    random_state=42, # reproducibility
    warm_start=True, # iterative training
    n_jobs=-1, # use maximum processors
)

for n_trees in range(50, 201, 50):
    rf.set_params(n_estimators=n_trees)
    rf.fit(x_train, y_train)

    y_pred = rf.predict(x_val)
    
    print(f"Trees: {n_trees}")
    print("Valiadation set results:")
    validation_results = helper_functions.evaluate_metrics(y_val, y_pred)
    print(validation_results.to_string())
    print("-" * 50)

    joblib.dump(rf, f"../models/random_forest/random_forest_{n_trees}_20_depth.joblib")

Trees: 50
Valiadation set results:
        IoU (Jaccard) Precision  Recall F1-Score
Class 0        86.02%    98.89%  86.86%   92.48%
Class 1        35.07%    36.72%  88.67%   51.93%
--------------------------------------------------
Trees: 100
Valiadation set results:
        IoU (Jaccard) Precision  Recall F1-Score
Class 0        86.08%     98.9%  86.91%   92.52%
Class 1        35.19%    36.83%  88.73%   52.06%
--------------------------------------------------
Trees: 150
Valiadation set results:
        IoU (Jaccard) Precision  Recall F1-Score
Class 0        86.11%     98.9%  86.95%   92.54%
Class 1        35.24%    36.89%  88.72%   52.11%
--------------------------------------------------
Trees: 200
Valiadation set results:
        IoU (Jaccard) Precision  Recall F1-Score
Class 0        86.11%     98.9%  86.94%   92.54%
Class 1        35.23%    36.88%  88.72%   52.11%
--------------------------------------------------


# Evaluate on training data to detect overfitting/underfitting

In [6]:
y_pred = rf.predict(x_train)
training_results = helper_functions.evaluate_metrics(y_train, y_pred)
print(training_results.to_string())

        IoU (Jaccard) Precision  Recall F1-Score
Class 0        87.08%    99.64%  87.36%    93.1%
Class 1        39.01%    39.59%  96.35%   56.12%


# Investigate if tree depth is the limiting factor for the random forest

In [7]:
tree_depths = [tree.get_depth() for tree in rf.estimators_]

print(f"Average depth: {np.mean(tree_depths):.2f}")
print(f"Minimum depth: {np.min(tree_depths)}")
print(f"Maximum depth: {np.max(tree_depths)}")
print(f"Std depth: {np.std(tree_depths):.2f}")

Average depth: 20.00
Minimum depth: 20
Maximum depth: 20
Std depth: 0.00


Analysis:
1. For the informal class, with 200 trees, the training `IoU` and `F1` scores are `39.01%` and `56.12%` respectively, while the valiadation `IoU` and `F1` scores are `35.23%` and `52.11%` respectively`. The relatively small gap suggests that the model is not simply memorizing the training pixels. If the model were strongly overfitting, we would expect substantially higher training performance than testing performance.

2. Increasing the forest from `50` to `200` trees only changes validation data informal class `IoU` from `35.07%` to `35.23%` and `F1` from `51.93%` to `52.11%`. This indicates that the ensemble has already stabilized at around `50` trees. Therefore, increasing n_estimators is unlikely to address the model's performance limitation.

3. On the validation set, with 200 trees, informal class precision is only `36.88%`, compared with a recall of `88.72%`, meaning that the model identifies most actual informal-settlement pixels but also incorrectly classifies many non-settlement pixels as settlements, leading to a high amount of false positives. High precision and recall scores for the formal class indicate that the model is highly accurate at marking formal settlements.

4. Overall, the model appears to be underfitting based on the following evidence:

    4.1. The similarity between training and validation metrics.

    4.2. The rapid plateau in performance as the number of trees in the random forest model are increased suggesting that the current Random Forest does not have sufficient capacity to fully model the data.

    4.3. The fact that all trees reached the maximum depth, meaning that the random forest model is being limited by the depth limit.

    Based on this evidence, the next notebook will include experiments involving increasing the maximum depth of trees in the random forest